# Istraživačka analiza podataka (EDA) – UNSW-NB15
**SIR1: Primena mašinskog učenja u detekciji mrežnih napada**

Ovaj notebook sprovodi analizu zvaničnih trening/test datoteka (bez ikakve transformacije
koja bi koristila test podatke) i demonstrira pipeline za preprocesiranje. Pokrenuti iz korenskog direktorijuma projekta ili iz `notebooks/`.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..') if os.path.basename(os.getcwd()) == 'notebooks' else os.path.abspath('.'))
import pandas as pd, numpy as np
from IPython.display import display, Image
from src import config, eda, visualization as viz
from src.data_loader import load_train_test, get_xy, load_feature_descriptions
from src.preprocessing import build_preprocessor, get_feature_names, log_transformed_columns
pd.set_option('display.max_columns', 60)

## 1. Učitavanje i dimenzije

In [ ]:
train, test = load_train_test()
print('Trening:', train.shape, '| Test:', test.shape)
train.head()

## 2. Kolone i tipovi podataka, nedostajuće vrednosti

In [ ]:
cols = eda.dimensions_and_types(train, test)
display(cols)
print('Ukupno nedostajućih (trening/test):', train.isna().sum().sum(), test.isna().sum().sum())
print('Beskonačne vrednosti:', int(np.isinf(train.select_dtypes('number')).sum().sum() + np.isinf(test.select_dtypes('number')).sum().sum()))

## 3. Duplirani zapisi i preklapanje trening/test
Kolona `id` je redni broj, pa se duplikati proveravaju bez nje.

In [ ]:
print(eda.duplicates_summary(train, test))
print(eda.overlap_summary(train, test))

## 4. Raspodela klasa i kategorija napada

In [ ]:
print(train['label'].value_counts().to_string()); print(test['label'].value_counts().to_string())
print(pd.concat([train['attack_cat'].value_counts().rename('trening'), test['attack_cat'].value_counts().rename('test')], axis=1))
display(Image(str(viz.plot_label_distribution(train, test))))
display(Image(str(viz.plot_attack_distribution(train, test))))

## 5. Numeričke i kategorijske promenljive

In [ ]:
X, y = get_xy(train, task='binary')
num = X.select_dtypes('number').columns.tolist(); cat = [c for c in X.columns if c not in num]
print(len(num), 'numeričkih;', len(cat), 'kategorijskih:', cat)
display(eda.categorical_summary(train, test))  # napomena: ACC i CLO postoje samo u test skupu
for c in cat: display(Image(str(viz.plot_categorical_vs_label(train, c))))

## 6. Asimetrija raspodela i korelacije

In [ ]:
sk = eda.skewness_table(train); display(sk.head(10))
display(Image(str(viz.plot_feature_distributions(train, ['dur','sbytes','dbytes','sload','rate','sttl']))))
display(Image(str(viz.plot_correlation_heatmap(train))))

## 7. Preprocesiranje (bez curenja podataka)
Preprocesor se uči (`fit`) samo na trening skupu; test skup prolazi samo kroz `transform`.

In [ ]:
X_tr, y_tr = get_xy(train, 'binary'); X_te, y_te = get_xy(test, 'binary')
prep = build_preprocessor(X_tr)
A = prep.fit_transform(X_tr)      # fit SAMO na treningu
B = prep.transform(X_te)          # test: samo transform
print('Trening matrica:', A.shape, '| Test matrica:', B.shape)
print('Kolone sa log1p (odredjeno na treningu):', log_transformed_columns(prep))
print('Nepoznate kategorije u testu (state: ACC, CLO) -> sve nule u OHE kolonama state_*')

## 8. Priprema za višeklasnu klasifikaciju (budući rad)
`get_xy(df, task='multiclass')` koristi `attack_cat` kao cilj i odbacuje `label`. Višeklasni eksperiment nije deo ovog rada.

In [ ]:
Xm, ym = get_xy(train, task='multiclass')
assert 'label' not in Xm.columns and 'attack_cat' not in Xm.columns
print(ym.value_counts().to_string())